# i2ce_llm_view.ipynb — LLM 改写文档视图塔（论文 Table 5 第三行）

User (2026-07-20): 训练 `wcle_i2ce_icetf` @512 但文档视图源换为逐句忠实
LLM 改写（`--wiki-src llm` → 塔名后缀 `_wllm`,语料 wiki_llm_views.npz）。
与已有的有文档塔（wiki_clean,.926/.657）和无文档塔（nodoc,.912/.603）
构成三方消融:文档溢价集中 noname,本塔回答"LLM 改写防火墙的代价"。
2000ep,ZS-only,rvsel。~5.6G,单卡 ~3h。AUTO-STOPS。


In [ ]:
# constants
import os

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_out"

ARM, CAP, EPOCHS = "wcle_i2ce_icetf", 512, 2000
WIKI_SRC = "llm"                       # -> tower name suffix _wllm
os.makedirs(OUT_DIR, exist_ok=True)
print(f"tower: w9_{ARM}_wllm @g{CAP} / {EPOCHS}ep")


In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy
        break
import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")


In [ ]:
# Stage the corpus into RAM (llm views not needed).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "wiki_llm_views.npz", "sp_raw_views.npz",
            "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)


In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# Run the single tower (done marker = ep{EPOCHS} npz with _wllm suffix).
import os, subprocess, threading, time
from pathlib import Path

cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
gpus = J.detect_gpus()
nm = J.fs_label(ARM, CAP, False, 0, "clean", 16) + "_wllm"
if (Path(OUT_DIR) / f"tower_{nm}_fp_ep{EPOCHS}.npz").exists():
    print(f"[skip] {nm} done")
else:
    stop_evt = threading.Event()
    threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True).start()
    ok = J.try_claim(cdir, nm)
    if not ok:
        print("[claim] fresh/held -- waiting 130s for the corpse window", flush=True)
        time.sleep(130)
        ok = J.try_claim(cdir, nm)
    if not ok:
        print(f"[claim] {nm} held elsewhere -- skipped", flush=True)
    else:
        cmd = ["python", "-u", J.FS_WORKER, "--data-dir", DATA_DIR, "--out-dir",
               OUT_DIR, "--repo", REPO, "--arm", ARM, "--anchor-cap", str(CAP),
               "--epochs", str(EPOCHS), "--ckpt-every", str(J.CKPT_EVERY),
               "--ckpt-seeds", str(J.FS_CKPT_SEEDS),
               "--topup-seeds", str(J.TOPUP_SEEDS),
               "--wiki-src", WIKI_SRC,
               "--full-pool", "--full-pool-path", FULL_POOL_PATH,
               "--claim-file", str(cdir / f"{nm}.claim")]
        print(f"[gpu{gpus[0]}] start {nm}", flush=True)
        t0 = time.time()
        with open(logd / f"{ARM}_wllm_g{CAP}.log", "w") as fh:
            p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                               env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpus[0]))
        if p.returncode != 0:
            (cdir / f"{nm}.claim").unlink(missing_ok=True)
        print(f"{'ok' if p.returncode == 0 else 'FAIL'} {nm} "
              f"[{(time.time() - t0) / 3600:.1f}h]", flush=True)
    stop_evt.set()


In [ ]:
# Readout: the document-view triple (paper Table 5).
import json
from pathlib import Path
def show(nm, lab):
    zb = Path(OUT_DIR) / f"zsbest_{nm}_fp.json"
    zp = Path(OUT_DIR) / f"zs_traj_{nm}_fp.json"
    if zb.exists():
        d = json.loads(zb.read_text())
        print(f"{lab:28s} ep{d['best_ep']:>4} neu {d['nm_neutral']:.3f} "
              f"h5 {d.get('h5_neutral', float('nan')):.3f} non {d['nm_noname']:.3f} "
              f"h5n {d.get('h5_noname', float('nan')):.3f} "
              f"tag {d['tag_neutral']:.3f}/{d['tag_noname']:.3f}")
    elif zp.exists():
        tr = json.loads(zp.read_text())
        pk = max(tr, key=lambda k: tr[k].get('nm_neutral', -9))
        d = tr[pk]
        print(f"{lab:28s} peak*{pk[2:]:>4} neu {d['nm_neutral']:.3f} "
              f"non {d['nm_noname']:.3f} tag {d['tag_neutral']:.3f}/{d['tag_noname']:.3f}"
              f"  (legacy traj)")
    else:
        print(f"{lab:28s} (pending)")
show("w9_wcle_i2ce_icetf", "doc view: wiki_clean")
show("w9_wcle_i2ce_icetf_wllm", "doc view: LLM rewrite")
show("w9_wcle_nodoc_i2ce_icetf", "no doc view")


In [ ]:
# AUTO-STOP the pod (results are on the network volume).
AUTO_STOP = True
if AUTO_STOP:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from VICReg_review import pod_selfstop
    pod_id, api_key, ctl = pod_selfstop.preflight("")
    pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    print("AUTO_STOP disabled -- stop the pod yourself.")
